<a href="https://colab.research.google.com/github/MaximSS1309/ML_SEREDOVSKIJ_MAXM/blob/main/%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%A1%D0%B5%D1%80%D0%B5%D0%B4%D0%BE%D0%B2%D1%81%D1%8C%D0%BA%D0%B8%D0%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** середовський максим
**Група:** 3-10
**Дата:** 18.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [ ]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.16.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [ ]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

unique_users = ratings_df['user_id'].nunique()
unique_movies = ratings_df['item_id'].nunique()

min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()

missing_values = ratings_df.isnull().sum()

print(f"Унікальних користувачів: {unique_users}")
print(f"Унікальних фільмів: {unique_movies}")
print(f"Рейтинг -> мін: {min_rating}, макс: {max_rating}, середнє: {mean_rating:.2f}")
print("\nПропущені значення за стовпцями:")
print(missing_values)

Унікальних користувачів: 943
Унікальних фільмів: 1682
Рейтинг -> мін: 1.0, макс: 5.0, середнє: 3.53

Пропущені значення за стовпцями:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**


Набір даних містить 943 унікальних користувачів та 1 682 унікальних фільмів (загалом 100 000 оцінок). Рейтинги знаходяться в межах від 1.0 до 5.0 балів із середнім значенням 3.53, що вказує на переважання позитивних оцінок над негативними. Пропущених значень у жодному зі стовпців не виявлено, тому набір даних є повним і готовим до подальшої побудови рекомендаційної моделі.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

train_df, test_df = sklearn_train_test_split(ratings_df, test_size=0.2, random_state=42)

print("Розмір навчальної вибірки (train_df):", train_df.shape)
print("Розмір тестової вибірки (test_df):", test_df.shape)

Розмір навчальної вибірки (train_df): (80000, 4)
Розмір тестової вибірки (test_df): (20000, 4)


In [ ]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

reader = Reader(rating_scale=data.reader.rating_scale)

# 2. Завантажуємо навчальну вибірку у формат Dataset бібліотеки Surprise
train_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)

# 3. Формуємо testset як список кортежів (user_id, item_id, rating)
testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))

# 4. Перевірка розмірів
print(f"Кількість записів у train_data: {len(train_data.raw_ratings)}")
print(f"Кількість записів у testset: {len(testset)}")

Кількість записів у train_data: 80000
Кількість записів у testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [ ]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._
Модель SVD
n_factors ([50, 100]): визначає кількість прихованих (латентних) факторів, що описують уподобання користувачів та властивості фільмів. Значення 50 і 100 дозволяють перевірити баланс між достатньою виразністю моделі та ризиком перенавчання на порівняно невеликому датасеті MovieLens 100k.
lr_all ([0.005, 0.010]): швидкість навчання (learning rate) при стохастичному градієнтному спуску. Базове значення 0.005 забезпечує стабільну збіжність, а 0.010 дозволяє перевірити, чи швидше оновлення ваг покращить точність за фіксовану кількість епох.
reg_all ([0.02, 0.05]): коефіцієнт L2-регуляризації, який запобігає перенавчанню на рідкісних оцінках. Перебір стандартного (0.02) та посиленого (0.05) штрафів допомагає знайти оптимальне узагальнення для розрідженої матриці рейтингів

Модель KNNBasic
k ([20, 40]): максимальна кількість найближчих сусідів, які враховуються при прогнозуванні оцінки. Порівнюються локальніша вибірка (20, більш чутлива до близьких смаків) та ширша (40, що дає більш усереднений та стабільний прогноз).
min_k ([1, 5]): мінімальна кількість сусідів, необхідна для обчислення прогнозу (якщо сусідів менше, повертається середній рейтинг). Значення 1 дозволяє робити прогноз навіть за мінімального збігу, тоді як 5 відсіює ненадійні прогнози на основі поодиноких збігів.
sim_options (name: ['cosine', 'pearson'], user_based: [False, True]): перевіряє дві популярні метрики близькості — косинусну схожість та кореляцію Пірсона (яка додатково враховує індивідуальний зсув оцінок користувачів), а також порівнює підходи User-based (True — пошук схожих користувачів) та Item-based (False — пошук схожих фільмів, що часто працює стабільніше, коли фільмів більше, ніж користувачів).

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
gs_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

gs_svd.fit(train_data)

print(f"Найкращий RMSE (SVD): {gs_svd.best_score['rmse']:.4f}")
print(f"Найкращі параметри за RMSE (SVD): {gs_svd.best_params['rmse']}")
print(f"Найкращий MAE (SVD): {gs_svd.best_score['mae']:.4f}")
print(f"Найкращі параметри за MAE (SVD): {gs_svd.best_params['mae']}")

Найкращий RMSE (SVD): 0.9433
Найкращі параметри за RMSE (SVD): {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Найкращий MAE (SVD): 0.7432
Найкращі параметри за MAE (SVD): {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}


In [ ]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
gs_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=2,
    joblib_verbose=1
)

gs_knn.fit(train_data)

print(f"\nНайкращий RMSE (KNNBasic): {gs_knn.best_score['rmse']:.4f}")
print(f"Найкращі параметри за RMSE (KNNBasic): {gs_knn.best_params['rmse']}")
print(f"Найкращий MAE (KNNBasic): {gs_knn.best_score['mae']:.4f}")
print(f"Найкращі параметри за MAE (KNNBasic): {gs_knn.best_params['mae']}")

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.



Найкращий RMSE (KNNBasic): 1.0260
Найкращі параметри за RMSE (KNNBasic): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Найкращий MAE (KNNBasic): 0.8138
Найкращі параметри за MAE (KNNBasic): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


[Parallel(n_jobs=2)]: Done  48 out of  48 | elapsed:  2.4min finished


In [ ]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
grid_search_svd = gs_svd
grid_search_knn = gs_knn

print("=== Результати GridSearchCV для SVD ===")
print(f"Найкращий RMSE: {grid_search_svd.best_score['rmse']:.4f}")
print(f"Параметри (RMSE): {grid_search_svd.best_params['rmse']}")
print(f"Найкращий MAE:  {grid_search_svd.best_score['mae']:.4f}")
print(f"Параметри (MAE):  {grid_search_svd.best_params['mae']}")

print("\n=== Результати GridSearchCV для KNNBasic ===")
print(f"Найкращий RMSE: {grid_search_knn.best_score['rmse']:.4f}")
print(f"Параметри (RMSE): {grid_search_knn.best_params['rmse']}")
print(f"Найкращий MAE:  {grid_search_knn.best_score['mae']:.4f}")
print(f"Параметри (MAE):  {grid_search_knn.best_params['mae']}")

=== Результати GridSearchCV для SVD ===
Найкращий RMSE: 0.9433
Параметри (RMSE): {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Найкращий MAE:  0.7432
Параметри (MAE):  {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

=== Результати GridSearchCV для KNNBasic ===
Найкращий RMSE: 1.0260
Параметри (RMSE): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Найкращий MAE:  0.8138
Параметри (MAE):  {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**
За результатами 3-блокової кросвалідації на навчальній вибірці модель SVD продемонструвала суттєво вищу точність прогнозування порівняно з KNNBasic, забезпечивши менші значення помилок як за метрикою RMSE (0.9433 проти 1.0260), так і за MAE (0.7432 проти 0.8138). Це зумовлено тим, що матрична факторизація (SVD) ефективніше виявляє приховані фактори вподобань користувачів на розріджених даних, ніж базовий метод пошуку найближчих сусідів.   Найкращими гіперпараметрами для SVD (за обома метриками) стали: кількість факторів n_factors = 50, швидкість навчання lr_all = 0.01 та посилена регуляризація reg_all = 0.05. Для KNNBasic оптимальною виявилася конфігурація з більшою кількістю сусідів k = 40, мінімальним порогом min_k = 1, косинусною метрикою схожості ('cosine') та підходом на основі схожості користувачів ('user_based': True).

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

# 6.2 Будуємо повну навчальну вибірку
trainset = train_data.build_full_trainset()

# 6.3 Ініціалізуємо та навчаємо остаточні моделі
best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)

print("Обидві моделі успішно навчені на повній навчальній вибірці!")
print("Параметри best_svd:", best_svd_params)
print("Параметри best_knn:", best_knn_params)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Обидві моделі успішно навчені на повній навчальній вибірці!
Параметри best_svd: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Параметри best_knn: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# 7.2 Обчислюємо метрики RMSE та MAE для обох моделей
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print("=== Фінальні результати на тестовій вибірці ===")
print(f"SVD      -> RMSE: {svd_rmse:.4f}, MAE: {svd_mae:.4f}")
print(f"KNNBasic -> RMSE: {knn_rmse:.4f}, MAE: {knn_mae:.4f}")

=== Фінальні результати на тестовій вибірці ===
SVD      -> RMSE: 0.9192, MAE: 0.7234
KNNBasic -> RMSE: 1.0120, MAE: 0.7984


In [ ]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
})

results = results.sort_values(by='RMSE', ascending=True).reset_index(drop=True)
display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0120 | 0.7984 |

**Порівняння з CV:**
Порівняння з CV: Оцінки на незалежній тестовій вибірці не мають суттєвих розбіжностей із результатами 3-блокової кросвалідації та навіть виявилися трохи кращими для обох моделей:   Для SVD похибка RMSE знизилася з 0.9433 (CV) до 0.9192 (Test), а MAE — з 0.7432 до 0.7234.   Для KNNBasic похибка RMSE покращилася з 1.0260 (CV) до 1.0120 (Test), а MAE — з 0.8138 до 0.7984

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:

# 8.1 Обираємо найкращу модель за результатами кросвалідації (CV RMSE)
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    best_model_name = "SVD"
else:
    best_model = best_knn
    best_model_name = "KNNBasic"

print(f"Обрана модель за найкращим CV RMSE: {best_model_name}")

# 8.2 Перевіряємо наявність користувача у trainset та отримуємо його внутрішній ID
try:
    inner_uid = trainset.to_inner_uid(user_id)

    # 8.3 Знаходимо множину фільмів, які користувач уже оцінив у навчальній вибірці
    rated_inner_iids = {iid for (iid, _) in trainset.ur[inner_uid]}

    # Формуємо список неоцінених фільмів (переводимо внутрішні ID у зовнішні raw ID)
    unrated_raw_iids = [
        trainset.to_raw_iid(inner_iid)
        for inner_iid in trainset.all_items()
        if inner_iid not in rated_inner_iids
    ]

    # 8.4 Прогнозуємо рейтинги для всіх неоцінених фільмів
    predictions = [
        (raw_iid, best_model.predict(uid=user_id, iid=raw_iid).est)
        for raw_iid in unrated_raw_iids
    ]

    # Сортуємо за спаданням прогнозованого рейтингу та формуємо DataFrame із топ-10
    top_10_df = (
        pd.DataFrame(predictions, columns=['item_id', 'predicted_rating'])
        .sort_values(by='predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    print(f"Топ-10 рекомендацій для користувача {user_id}:")
    display(top_10_df)

except ValueError:
    print(f"Користувач із ID '{user_id}' відсутній у навчальній вибірці (проблема холодного старту).")

Обрана модель за найкращим CV RMSE: SVD
Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010




## 9. Перевірка виконання

Позначте після реалізації:
- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:* За результатами 3-блокової кросвалідації найкращою моделлю обрано SVD, оскільки вона показала нижчі значення похибок (CV RMSE = 0.9433, CV MAE = 0.7432) порівняно з KNNBasic (CV RMSE = 1.0260, CV MAE = 0.8138) завдяки здатності матричної факторизації краще виявляти приховані вподобання на розріджених даних. Оптимальними гіперпараметрами для SVD стали n_factors = 50, lr_all = 0.01, reg_all = 0.05, а для KNNBasic — k = 40, min_k = 1, метрика 'cosine' та режим 'user_based': True. На незалежній тестовій вибірці фінальні моделі продемонстрували результати RMSE = 0.9192 та MAE = 0.7234 для SVD і RMSE = 1.0120 та MAE = 0.7984 для KNNBasic. Різниця між оцінками кросвалідації та тестування є незначною (метрики на тесті навіть трохи покращилися за рахунок навчання моделей на всіх 80% навчальних даних замість 2/3 під час CV), що підтверджує відсутність перенавчання. Головним обмеженням побудованої колаборативної рекомендаційної системи є проблема «холодного старту»: модель не може формувати персоналізовані прогнози для нових користувачів або нових фільмів, які відсутні в навчальній вибірці та не мають історії оцінок. Для усунення цього недоліку в реальних системах колаборативну фільтрацію доповнюють контентною фільтрацією (за жанрами чи описом фільмів) або рекомендаціями найпопулярніших позицій.
---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**